# Semantic mode slots on DeMo (Colab GPU)
Run top to bottom. Works in the browser or in VS Code with the Colab extension (kernel: Colab, GPU / L4).
- **One restart** after S2. Data lives on the Colab machine (`/content`) and is lost when the runtime is deleted: then run everything again (cells skip files that already exist on Drive).
- Results to send back are marked **Send**.

## S. Setup
### S1. Drive and code

In [1]:
import os, sys, torch
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/thesis_demo'
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
print(sys.version.split()[0], torch.__version__, 'GPU:', torch.cuda.is_available())

REPO, BRANCH = 'https://github.com/Nohahamid/DeMo.git', 'named-intentions'
if not os.path.exists('DeMo'):
    !git clone -b {BRANCH} {REPO} DeMo
%cd DeMo
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}
!git log -1 --format='code version: %h %s'
!grep -q HAS_CUDA_KERNELS third_party/mamba/mamba_ssm/ops/selective_scan_interface.py && echo 'OK: Mamba fix is in the code' || echo 'PROBLEM: old code, send me this output'

Mounted at /content/drive
3.13.16 2.14.1+cu130 GPU: True
/content/drive/MyDrive/thesis_demo/DeMo
code version: c00235c Colab notebook: R1 saves processed data to Drive, R0 restores it after a disconnect
OK: Mamba fix is in the code


### S2. Packages and Mamba (about 2 min; red numpy warnings are fine)

In [2]:
!pip install -q av2==0.2.1 hydra-core==1.3.2 pytorch-lightning torchmetrics==0.10.2 timm einops rich gdown tensorboard "numpy<2" ninja packaging wheel
!pip uninstall -y -q mamba_ssm causal_conv1d
%cd third_party/mamba
!MAMBA_SKIP_CUDA_BUILD=TRUE MAMBA_FORCE_BUILD=TRUE pip install -q . --no-build-isolation --no-deps --no-cache-dir
%cd ../..
print('Done. Now: Runtime -> Restart session, then run S3.')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.11.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.11.1 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.11.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.10.15 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, bu

### ⟳ Runtime → Restart session, then continue with S3

### S3. After the restart

In [1]:
import os, sys, torch
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/thesis_demo'
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
print(sys.version.split()[0], torch.__version__, 'GPU:', torch.cuda.is_available())

os.chdir(f'{WORK}/DeMo')
import mamba_ssm.ops.selective_scan_interface as ssi
from mamba_ssm.modules.mamba_simple import Mamba
print('CUDA kernels:', ssi.HAS_CUDA_KERNELS, '(False is expected: pure-PyTorch version)')
m = Mamba(d_model=128, bimamba=True).cuda()
print(m(torch.randn(2, 60, 128, device='cuda')).shape, '<- expect torch.Size([2, 60, 128])')
!git pull -q origin named-intentions && git log -1 --format='code version: %h %s'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
3.13.16 2.14.1+cu130 GPU: True


/usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/selective_scan_interface.py:175: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/selective_scan_interface.py:244: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/selective_scan_interface.py:312: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.13/dist-packages/mamba_ssm/ops/selective_scan_interface.py:385: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.13/dist-packages/

CUDA kernels: False (False is expected: pure-PyTorch version)
torch.Size([2, 60, 128]) <- expect torch.Size([2, 60, 128])
code version: c00235c Colab notebook: R1 saves processed data to Drive, R0 restores it after a disconnect


In [2]:
import socket, time, re, urllib.request, urllib.parse, concurrent.futures, pathlib
from tqdm.auto import tqdm
socket.setdefaulttimeout(30)
def download(root, split, n):
    """First n AV2 motion-forecasting scenarios of a split into root/split (skips files already there)."""
    base = 'https://argoverse.s3.us-east-1.amazonaws.com'
    keys, token = [], None
    while len(keys) < 2 * n:
        url = f'{base}/?list-type=2&prefix=datasets/av2/motion-forecasting/{split}/&max-keys=1000'
        if token: url += '&continuation-token=' + urllib.parse.quote(token)
        xml = urllib.request.urlopen(url).read().decode()
        keys += re.findall(r'<Key>([^<]+)</Key>', xml)
        m = re.search(r'<NextContinuationToken>([^<]+)</NextContinuationToken>', xml)
        if not m: break
        token = m.group(1)
    keys = keys[:2 * n]   # 2 files per scenario: parquet + map json
    def fetch(k):
        out = pathlib.Path(root) / split / pathlib.Path(k).parent.name / pathlib.Path(k).name
        if out.exists(): return
        out.parent.mkdir(parents=True, exist_ok=True)
        for attempt in range(3):
            try:
                tmp = out.with_suffix(out.suffix + '.tmp')
                urllib.request.urlretrieve(f'{base}/{k}', tmp); tmp.rename(out); return
            except Exception:
                time.sleep(2)
    with concurrent.futures.ThreadPoolExecutor(32) as ex: list(tqdm(ex.map(fetch, keys), total=len(keys)))
    print(len(list(pathlib.Path(root, split).iterdir())), split, 'scenarios in', root)

## R0. Fast restart (only if you saved with R1 before)
If `{WORK}/data_cache.tar` exists, this restores the processed data in about 5 min; then **skip V, H1–H2 and H3** and go to H4.

In [ ]:
import os
if os.path.exists(f'{WORK}/data_cache.tar'):
    !tar -xf {WORK}/data_cache.tar -C /content
    !ls /content/data/DeMo_processed/train | wc -l
    !ls /content/val5k/data/DeMo_processed/val | wc -l
else:
    print('No cache yet: run V, H1-H2, H3, then R1.')

## V. 5,000 val scenarios (about 15 min)

In [3]:
download('/content/av2_val5k', 'val', 5000)
!mkdir -p /content/val5k
%cd /content/val5k
!python {WORK}/DeMo/preprocess.py --data_root /content/av2_val5k -p
%cd {WORK}/DeMo
!ls /content/val5k/data/DeMo_processed/val | wc -l
import os
if not os.path.exists('data/intention_labels_val5k/val.csv'):
    !python intention_labels.py --data_root /content/av2_val5k --split val --out data/intention_labels_val5k/val.csv

  0%|          | 0/10000 [00:00<?, ?it/s]

5000 val scenarios in /content/av2_val5k
/content/val5k
0it [00:00, ?it/s]
100% 5000/5000 [01:04<00:00, 77.39it/s]
0it [00:00, ?it/s]
/content/drive/MyDrive/thesis_demo/DeMo
5000


## H0. Baseline column for plain DeMo (1 min)
**Send** the output. Needs `fusion_val.npz` from G1 (on Drive).

In [4]:
!python fusion_intent.py diagnose --val {WORK}/fusion_val.npz

5000 val scenarios, miss = final error > 2.0 m
overall: minFDE6 1.258 m, minADE6 0.650 m, MR6 15.7%, semantic diversity@6 1.26 distinct maneuvers (vehicles 1.28)

1. Per true maneuver (A + B + C + OK = 100%)
maneuver           n  Cov@6   Top1  minFDE6    MR6  div6  condFDE  topFDE  A gen B rank C geom     OK
STRAIGHT        3541  99.6%  95.3%     1.07  11.0%  1.18     1.13    3.55   0.4%   4.2%  53.7%  41.6%
LEFT_TURN        415  93.7%  81.4%     2.33  36.6%  1.47     2.03    4.80   6.3%  12.3%  63.4%  18.1%
RIGHT_TURN       333  92.8%  76.9%     2.35  35.1%  1.54     2.06    4.02   7.2%  15.9%  54.1%  22.8%
LANE_CHANGE_L     60  53.3%  43.3%     2.56  51.7%  1.37     3.21    5.19  46.7%  10.0%  35.0%   8.3%
LANE_CHANGE_R     70  42.9%  35.7%     2.29  55.7%  1.30     1.86    4.39  57.1%   7.1%  17.1%  18.6%
U_TURN             1 100.0%   0.0%     1.95   0.0%  2.00    15.12     nan   0.0% 100.0%   0.0%   0.0%
STOP             173  88.4%  82.1%     0.49   7.5%  2.14     0.21    0.30  11.

## H1–H2. 50k training scenarios: download and preprocess (about 1 h)
The same 50k as before, so the labels on Drive still match.

In [ ]:
download('/content/av2_raw', 'train', 50000)
%cd /content
!python {WORK}/DeMo/preprocess.py --data_root /content/av2_raw -p
%cd {WORK}/DeMo
!ls /content/data/DeMo_processed/train | wc -l

  0%|          | 0/100000 [00:00<?, ?it/s]

## H3. Lane-graph slots (about 15 min)
Prints, per maneuver, the share of scenarios where the map allows it.

In [ ]:
!python semantic_slots.py --data_root /content/av2_raw --split train --out data/semantic_slots/train.csv
!python semantic_slots.py --data_root /content/av2_val5k --split val --out data/semantic_slots/val.csv

## R1. Save the processed data to Drive (once, about 10 min)
So a disconnect no longer costs an hour: next time run S1–S3, then R0.

In [ ]:
!du -sh /content/data/DeMo_processed/train /content/val5k/data/DeMo_processed/val /content/av2_val5k
!tar -cf {WORK}/data_cache.tar -C /content data/DeMo_processed/train val5k/data/DeMo_processed/val av2_val5k
!ls -lh {WORK}/data_cache.tar

## H4. Train and 5k val side by side

In [ ]:
!rm -rf /content/data/DeMo_processed/val
!ln -s /content/val5k/data/DeMo_processed/val /content/data/DeMo_processed/val
!mkdir -p data/intention_labels_sem
!cp data/intention_labels/train.csv data/intention_labels_sem/train.csv
!cp data/intention_labels_val5k/val.csv data/intention_labels_sem/val.csv
!ls /content/data/DeMo_processed/val | wc -l

## H5a. Smoke test (about 5 min)
Should print `train_scope=decoder: training ... parameters only.`
**Send** the speed from the progress bar (`it/s` or `s/it`).

In [ ]:
!python train.py datamodule=av2 model=model_forecast gpus=1 sync_bn=false \
    batch_size=32 num_workers=8 \
    datamodule.target.data_root=/content/data/DeMo_processed \
    pretrained_weights=ckpts/DeMo.ckpt num_intentions=0 \
    semantic_slots=true slot_dir=data/semantic_slots \
    intention_dir=data/intention_labels_sem train_scope=decoder \
    "oversample={1:2,2:2,3:5,4:5}" \
    epochs=1 warmup_epochs=0 lr=0.0003 limit_train_batches=50 limit_val_batches=20

## H5b. Full training
**Send** the last validation table.

In [ ]:
!python train.py datamodule=av2 model=model_forecast gpus=1 sync_bn=false \
    batch_size=32 num_workers=8 \
    datamodule.target.data_root=/content/data/DeMo_processed \
    pretrained_weights=ckpts/DeMo.ckpt num_intentions=0 \
    semantic_slots=true slot_dir=data/semantic_slots \
    intention_dir=data/intention_labels_sem train_scope=decoder \
    "oversample={1:2,2:2,3:5,4:5}" \
    epochs=3 warmup_epochs=1 lr=0.0003 save_top_k=2

## H6. Diagnosis of the new model
**Send** the whole output.

In [ ]:
import glob, os
CK = max(glob.glob('outputs/**/checkpoints/last.ckpt', recursive=True), key=os.path.getmtime)
print(CK)
!python fusion_intent.py extract --ckpt {CK} --split val \
    --data_root /content/val5k/data/DeMo_processed --raw_root /content/av2_val5k \
    --intention_dir data/intention_labels_val5k --slot_dir data/semantic_slots --out {WORK}/sem_val.npz
!python fusion_intent.py diagnose --val {WORK}/sem_val.npz